1. **Ray actor**：一个带状态的 worker process，通过 RPC 暴露方法。
2. **Named actor**：一个注册了稳定名字的 actor，后续代码可以按名字重新查找它。
3. **Actor handle**：Python 侧持有的引用对象，它代表当前进程手里的一份 actor 引用。
4. **Actor owner**：最初创建这个 actor 的那个 worker 或 driver。默认情况下，owner 的生命周期对 actor 是否继续存活有决定性影响。
5. **Detached actor**：一种特殊的生命周期模式，即使创建它的 driver 或临时 handle 消失了，actor 仍然可以继续存活。

这里容易误解、但又最关键的一点是：

> **named actor 并不会自动变成 detached actor**。

所以 `name="sandbox-execution-pool"` 只表示这个 actor 可以被按名字发现，**并不** 保证它在最后一个 handle 消失后还能继续存活。

In [1]:
import gc
import json
import time
from pprint import pprint

import ray
import requests

/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-23 15:00:54,053	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
SANDBOX_URL = "http://10.1.123.37:8080/run_code"

def reset_ray(namespace: str):
    if ray.is_initialized():
        ray.shutdown()
    return ray.init(namespace=namespace, ignore_reinit_error=True, log_to_driver=False)


def call_run_code(code: str, *, sandbox_url: str = SANDBOX_URL, run_timeout: int = 10, memory_limit_mb: int = 256):
    payload = {
        "compile_timeout": run_timeout,
        "run_timeout": run_timeout,
        "code": code,
        "stdin": "",
        "memory_limit_MB": memory_limit_mb,
        "language": "python",
        "files": {},
        "fetch_files": [],
    }
    resp = requests.post(
        sandbox_url,
        headers={"Content-Type": "application/json", "Accept": "application/json"},
        data=json.dumps(payload),
        timeout=run_timeout + 10,
    )
    resp.raise_for_status()
    return resp.json()


def actor_exists(name: str) -> bool:
    try:
        ray.get_actor(name)
        return True
    except ValueError:
        return False


def kill_actor_if_exists(name: str):
    try:
        ray.kill(ray.get_actor(name), no_restart=True)
    except ValueError:
        pass

In [4]:
sandbox_result = call_run_code("total = sum(i * i for i in range(5))\nprint(total)")
pprint(sandbox_result)
print("stdout:", repr(sandbox_result["run_result"]["stdout"]))

{'compile_result': None,
 'executor_pod_name': None,
 'files': {},
 'message': '',
 'run_result': {'execution_time': 0.09086728096008301,
                'return_code': 0,
                'status': 'Finished',
                'stderr': '',
                'stdout': '30\n'},
 'status': 'Success'}
stdout: '30\n'


In [5]:
sum(i * i for i in range(5))

30

### 创建 ray actor

In [6]:
@ray.remote
class ExecutionWorker:
    def __init__(self, sandbox_url: str):
        self.sandbox_url = sandbox_url

    def ping(self):
        return "pong"

    def execute_python(self, code: str, run_timeout: int = 10):
        return call_run_code(code, sandbox_url=self.sandbox_url, run_timeout=run_timeout)


def init_execution_pool(actor_name: str, *, detached: bool, sandbox_url: str = SANDBOX_URL):
    options = {
        "name": actor_name,
        "get_if_exists": True,
    }
    if detached:
        options["lifetime"] = "detached"
    return ExecutionWorker.options(**options).remote(sandbox_url)


class SandboxTool:
    def __init__(self, actor_name: str, *, detached: bool, sandbox_url: str = SANDBOX_URL):
        self.actor_name = actor_name
        self.execution_pool = init_execution_pool(actor_name, detached=detached, sandbox_url=sandbox_url)
        self.tool_schema = {
            "type": "function",
            "function": {
                "name": "code_interpreter",
                "description": "Execute Python code in the remote sandbox.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "code": {"type": "string"},
                    },
                    "required": ["code"],
                },
            },
        }

    def get_openai_tool_schema(self):
        return self.tool_schema

    def execute(self, code: str):
        return ray.get(self.execution_pool.execute_python.remote(code))

### 正常流程

- 创建一个 tool object
- 持续持有它内部的 actor handle
- 通过这个 actor 去调用远端 sandbox

只要还有活跃的 handle 存在，这个 actor 对外看起来就是一个稳定的 RPC service。

In [7]:
reset_ray("normal-flow")
tool = SandboxTool(actor_name="normal-pool", detached=False)

2026-04-23 15:04:26,546	INFO worker.py:2012 -- Started a local Ray instance.
/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


In [11]:
tool.get_openai_tool_schema()

{'type': 'function',
 'function': {'name': 'code_interpreter',
  'description': 'Execute Python code in the remote sandbox.',
  'parameters': {'type': 'object',
   'properties': {'code': {'type': 'string'}},
   'required': ['code']}}}

In [8]:
print("ping:", ray.get(tool.execution_pool.ping.remote()))

ping: pong


In [12]:
result = tool.execute("print('normal flow works')")
print(result["run_result"]["stdout"])
print("actor exists while handle is alive:", actor_exists("normal-pool"))

kill_actor_if_exists("normal-pool")
ray.shutdown()

normal flow works

actor exists while handle is alive: True


### verl 中的异常


1. `RLHFDataset` 读取 tool config，但它的目的只是拿到 `tool_schema`。
2. 在构造 `SandboxFusionTool` 时，named actor `sandbox-execution-pool` 立刻被创建出来。
3. 这个“临时 schema 初始化”所在的进程，很可能成为该 actor 的 owner。
4. schema 提取完成后，这个临时 tool object 以及对应进程会很快结束自己的使命。
5. 后续真正做 rollout 的另一侧即使重新拿到了 handle，也不等于它成为了新的 owner。
6. 一旦最初的 owner 退出，而 actor 又不是 detached actor，那么 actor 仍然可能被 Ray 判定死亡。

所以更准确地说，根因不是单纯“最后一个 Python 变量没了”，而是 **actor lifetime / ownership 管理** 出了问题；sandbox HTTP 服务本身通常是正常的。

这也解释了为什么你的日志里会出现“all references were removed”或 owner 退出后 actor 死亡这一类信息。

In [17]:
reset_ray("gc-proof")

def schema_only_init(detached: bool, actor_name: str):
    temp_tool = SandboxTool(actor_name=actor_name, detached=detached)
    print("inside function ping:", ray.get(temp_tool.execution_pool.ping.remote()))
    return temp_tool.get_openai_tool_schema()

_ = schema_only_init(detached=False, actor_name="gc-proof-pool")
gc.collect()
time.sleep(0.2)

print("actor exists after the temporary tool disappears:", actor_exists("gc-proof-pool"))
ray.shutdown()

2026-04-23 15:33:34,504	INFO worker.py:2012 -- Started a local Ray instance.


inside function ping: pong
actor exists after the temporary tool disappears: False


- dataset 侧创建了一个临时 tool
- 这个临时 tool 创建了一个 named actor
- 后续只保留了 schema
- 临时 tool 消失了
- 因为它 **不是** detached actor，所以对应的 named actor 也一起消失了
    - 如果它是默认的 attached actor
        - owner 还活着，它就活着
        - owner 死了 / 退出了，它也可能一起死
    - 如果它是 detached actor
        - owner 退出了，它还可以继续活
        - 后面的别的进程还能重新连上它

In [16]:
reset_ray("tutorial-downstream-failure")

tool = SandboxTool(actor_name="failure-pool", detached=False)
print(tool.execute("print('before failure')")["run_result"]["stdout"])

# Force the same class of downstream failure deterministically.
# In real training job, Ray GC triggered the death instead of ray.kill().
ray.kill(tool.execution_pool, no_restart=True)

after_failure = None
try:
    after_failure = tool.execute("print('after failure')")
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:600])
finally:
    ray.shutdown()

after_failure

2026-04-23 15:20:10,110	INFO worker.py:2012 -- Started a local Ray instance.


before failure

ActorDiedError
The actor died unexpectedly before finishing this task.
	class_name: ExecutionWorker
	actor_id: 88b0cfedd94e0d5c0eaa4c3801000000
	pid: 184370
	name: failure-pool
	namespace: tutorial-downstream-failure
	ip: 10.2.160.10
The actor is dead because it was killed by `ray.kill`.


### verl issue rollout

### 修复方式：`lifetime="detached"`

“只初始化 schema”模式，但把 actor 创建成 detached actor。

预期现象：
- 临时 tool object 可以消失
- named actor 仍然存活
- 后续重新创建的 tool instance 仍然可以连回同一个 actor，并继续使用它

In [15]:
reset_ray("fix")

_ = schema_only_init(detached=False, actor_name="buggy-pool")
gc.collect()
time.sleep(0.2)
print("buggy actor survives schema-only init:", actor_exists("buggy-pool"))

_ = schema_only_init(detached=True, actor_name="fixed-pool")
gc.collect()
time.sleep(0.2)
print("detached actor survives schema-only init:", actor_exists("fixed-pool"))

fixed_tool = SandboxTool(actor_name="fixed-pool", detached=True)
fixed_result = fixed_tool.execute("print('detached actor still works')")
print(fixed_result["run_result"]["stdout"])

kill_actor_if_exists("fixed-pool")
ray.shutdown()

2026-04-23 15:10:15,348	INFO worker.py:2012 -- Started a local Ray instance.


inside function ping: pong
buggy actor survives schema-only init: False
inside function ping: pong
detached actor survives schema-only init: True
detached actor still works



### `verl` 里的最小修复

在 `verl/tools/sandbox_fusion_tools.py` 里，把 actor 创建逻辑从：

```python
ray.remote(ExecutionWorker).options(
    name="sandbox-execution-pool",
    get_if_exists=True,
    max_concurrency=num_workers,
).remote(...)
```

改成：

```python
ray.remote(ExecutionWorker).options(
    name="sandbox-execution-pool",
    get_if_exists=True,
    lifetime="detached",
    max_concurrency=num_workers,
).remote(...)
```

它之所以有效，是因为：
- dataset 侧那个临时 tool 即使消失，也不会顺带把 actor 带死
- 后面的 rollout 逻辑仍然可以重新连接同一个 named actor
- 这是一个局部且最小的修复
